# Medicare Denial Intelligence — Exploratory Data Analysis

Explores dbt **marts** built from CMS Medicare public data:

- `public_marts.dim_providers`
- `public_marts.fct_provider_spending`
- `public_marts.fct_utilization_by_specialty`

**Prerequisites:** Docker Postgres running, marts built (`dbt run --select marts`).

Set `POSTGRES_HOST=localhost` in `.env` when running locally.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

import os

engine = create_engine(
    "postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}".format(
        user=os.getenv("POSTGRES_USER", "denial_user"),
        password=os.getenv("POSTGRES_PASSWORD", ""),
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=os.getenv("POSTGRES_PORT", "5432"),
        db=os.getenv("POSTGRES_DB", "denial_db"),
    )
)


def q(sql: str, **params) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

sns.set_theme(style="whitegrid")
pd.options.display.float_format = lambda x: f"{x:,.4f}"

## 1. Dataset overview

In [ ]:
overview = q("""
SELECT 'dim_providers' AS table_name, COUNT(*) AS row_count
FROM public_marts.dim_providers
UNION ALL
SELECT 'fct_provider_spending', COUNT(*)
FROM public_marts.fct_provider_spending
UNION ALL
SELECT 'fct_utilization_by_specialty', COUNT(*)
FROM public_marts.fct_utilization_by_specialty
""")
overview

## 2. Implied withhold rate by specialty

Withhold rate = `1 - (Medicare payment / allowed amount)` — used here as a **denial proxy** when line-level denial codes are unavailable in public CMS files.

In [ ]:
specialty = q("""
SELECT
    specialty,
    year,
    SUM(provider_count) AS providers,
    SUM(service_line_count) AS service_lines,
    ROUND(AVG(avg_withhold_rate)::numeric, 4) AS avg_withhold_rate,
    ROUND(AVG(median_withhold_rate)::numeric, 4) AS median_withhold_rate
FROM public_marts.fct_utilization_by_specialty
WHERE specialty <> 'Unknown'
GROUP BY specialty, year
ORDER BY avg_withhold_rate DESC
LIMIT 20
""")
specialty

In [ ]:
plot_df = specialty.head(15).copy()
plot_df["label"] = plot_df["specialty"].str.slice(0, 40)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df, y="label", x="avg_withhold_rate", hue="year", ax=ax)
ax.set_xlabel("Average implied withhold rate")
ax.set_ylabel("")
ax.set_title("Top specialties by average withhold rate")
plt.tight_layout()
plt.show()

## 3. Geographic variation (state × year)

In [ ]:
by_state = q("""
SELECT
    state,
    year,
    ROUND(AVG(avg_withhold_rate)::numeric, 4) AS avg_withhold_rate,
    SUM(provider_count) AS providers
FROM public_marts.fct_utilization_by_specialty
WHERE state <> 'Unknown'
GROUP BY state, year
ORDER BY avg_withhold_rate DESC
LIMIT 15
""")
by_state

In [ ]:
pivot = by_state.pivot(index="state", columns="year", values="avg_withhold_rate")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd")
plt.title("Average withhold rate by state and year (top 15 states)")
plt.tight_layout()
plt.show()

## 4. Part D drug spending concentration

In [ ]:
top_drugs = q("""
SELECT
    COALESCE(drug_name, generic_name, 'Unknown') AS drug,
    year,
    COUNT(DISTINCT npi) AS prescriber_count,
    SUM(total_claim_count) AS claims,
    ROUND(SUM(total_drug_cost)::numeric, 2) AS total_medicare_cost
FROM public_marts.fct_provider_spending
GROUP BY 1, 2
ORDER BY total_medicare_cost DESC
LIMIT 20
""")
top_drugs

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_drugs = top_drugs.head(12).copy()
plot_drugs["label"] = plot_drugs["drug"].str.slice(0, 35)
sns.barplot(
    data=plot_drugs,
    y="label",
    x="total_medicare_cost",
    hue="year",
    ax=ax,
)
ax.set_xlabel("Total Medicare Part D cost (USD)")
ax.set_ylabel("")
ax.set_title("Highest-cost Part D drugs")
plt.tight_layout()
plt.show()

## 5. Provider dimension snapshot

In [ ]:
provider_mix = q("""
SELECT
    entity_type_label,
    state,
    COUNT(*) AS provider_count
FROM public_marts.dim_providers
WHERE state IS NOT NULL
GROUP BY entity_type_label, state
ORDER BY provider_count DESC
LIMIT 15
""")
provider_mix

## 6. ML feature preview (next step)

A future classifier can label providers/service lines with **high withhold risk** using features such as:

- specialty, state, place of service
- average allowed vs. paid amounts
- claim volume and beneficiary counts

Target idea: `high_withhold = implied_withhold_rate > specialty_median` at the provider level.

In [ ]:
ml_preview = q("""
WITH provider_rates AS (
    SELECT
        u.npi,
        u.cms_specialty,
        u.provider_state,
        u.year,
        AVG(u.implied_withhold_rate) AS avg_withhold_rate,
        COUNT(*) AS service_lines
    FROM public_intermediate.int_utilization_enriched u
    WHERE u.implied_withhold_rate IS NOT NULL
    GROUP BY 1, 2, 3, 4
),
specialty_medians AS (
    SELECT
        cms_specialty,
        year,
        percentile_cont(0.5) WITHIN GROUP (ORDER BY avg_withhold_rate) AS specialty_median
    FROM provider_rates
    GROUP BY 1, 2
)
SELECT
    p.npi,
    p.cms_specialty,
    p.provider_state,
    p.year,
    ROUND(p.avg_withhold_rate::numeric, 4) AS avg_withhold_rate,
    ROUND(s.specialty_median::numeric, 4) AS specialty_median,
    CASE WHEN p.avg_withhold_rate > s.specialty_median THEN 1 ELSE 0 END AS high_withhold_flag,
    p.service_lines
FROM provider_rates p
JOIN specialty_medians s
  ON p.cms_specialty = s.cms_specialty
 AND p.year = s.year
ORDER BY p.avg_withhold_rate DESC
LIMIT 20
""")
ml_preview